# HTTP y REST — Práctica

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/18_intro_a_apis/code/01_http_y_rest.ipynb)


## 1. Setup

Instalamos las librerías necesarias para hacer peticiones HTTP.

In [ ]:
%pip install requests httpx

In [ ]:
import requests
import httpx
import json

## 2. GET request básico

El método `GET` se usa para **obtener** información de un servidor.
Usaremos [httpbin.org](https://httpbin.org), un servicio público diseñado para probar peticiones HTTP.

In [ ]:
# Petición GET simple
respuesta = requests.get("https://httpbin.org/get")

# Código de estado (200 = éxito)
print("Código de estado:", respuesta.status_code)

In [ ]:
# Headers de la respuesta
print("Content-Type:", respuesta.headers["Content-Type"])
print("Server:", respuesta.headers.get("Server", "no especificado"))

In [ ]:
# Cuerpo de la respuesta como JSON
datos = respuesta.json()
print(json.dumps(datos, indent=2))

In [ ]:
# GET con parámetros de consulta (query params)
respuesta = requests.get(
    "https://httpbin.org/get",
    params={"curso": "fdd", "semestre": "p26"}
)

# Los parámetros se agregan a la URL automáticamente
print("URL final:", respuesta.url)
print("Args recibidos:", respuesta.json()["args"])

## 3. POST request

El método `POST` se usa para **enviar** datos al servidor, típicamente para crear un recurso nuevo.

In [ ]:
# POST con cuerpo JSON
payload = {
    "nombre": "Ana",
    "materia": "Fuentes de Datos",
    "calificacion": 95
}

respuesta = requests.post(
    "https://httpbin.org/post",
    json=payload  # automáticamente serializa a JSON y pone Content-Type
)

print("Código de estado:", respuesta.status_code)

In [ ]:
# Inspeccionamos lo que el servidor recibió
datos = respuesta.json()
print("Datos enviados (json):", datos["json"])
print("Content-Type enviado:", datos["headers"]["Content-Type"])

In [ ]:
# POST con datos de formulario (form-encoded)
respuesta = requests.post(
    "https://httpbin.org/post",
    data={"usuario": "estudiante01", "accion": "login"}
)

datos = respuesta.json()
print("Form data recibido:", datos["form"])
print("Content-Type:", datos["headers"]["Content-Type"])

## 4. PUT y DELETE

- `PUT` se usa para **actualizar** un recurso existente.
- `DELETE` se usa para **eliminar** un recurso.

In [ ]:
# PUT: actualizar un recurso
respuesta = requests.put(
    "https://httpbin.org/put",
    json={"nombre": "Ana", "calificacion": 98}  # calificación actualizada
)

print("Método:", respuesta.request.method)
print("Datos enviados:", respuesta.json()["json"])

In [ ]:
# DELETE: eliminar un recurso
respuesta = requests.delete("https://httpbin.org/delete")

print("Método:", respuesta.request.method)
print("Código de estado:", respuesta.status_code)

## 5. Headers y autenticación

Los headers HTTP permiten enviar metadatos con cada petición.
La autenticación comúnmente se envía en el header `Authorization`.

In [ ]:
# Headers personalizados
headers = {
    "X-Mi-Header": "valor-personalizado",
    "Accept": "application/json",
    "User-Agent": "ITAM-FDD-P26/1.0"
}

respuesta = requests.get("https://httpbin.org/headers", headers=headers)
print(json.dumps(respuesta.json()["headers"], indent=2))

In [ ]:
# Autenticación con Bearer Token (común en APIs modernas)
token_ficticio = "eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.ejemplo"

respuesta = requests.get(
    "https://httpbin.org/bearer",
    headers={"Authorization": f"Bearer {token_ficticio}"}
)

print("Código de estado:", respuesta.status_code)
print("Respuesta:", respuesta.json())

In [ ]:
# Autenticación básica (usuario y contraseña)
respuesta = requests.get(
    "https://httpbin.org/basic-auth/usuario/secreto",
    auth=("usuario", "secreto")
)

print("Autenticado:", respuesta.json()["authenticated"])
print("Usuario:", respuesta.json()["user"])

## 6. Códigos de status

Los códigos de estado HTTP indican el resultado de la petición:

| Rango | Significado |
|-------|------------|
| 2xx   | Éxito |
| 3xx   | Redirección |
| 4xx   | Error del cliente |
| 5xx   | Error del servidor |

In [ ]:
# httpbin.org puede devolver cualquier código de estado
codigos = [200, 201, 301, 404, 418, 500]

for codigo in codigos:
    respuesta = requests.get(
        f"https://httpbin.org/status/{codigo}",
        allow_redirects=False  # evitar seguir redirecciones automáticas
    )
    print(f"  {codigo}: {respuesta.reason}")

In [ ]:
# Verificar si la petición fue exitosa
respuesta_ok = requests.get("https://httpbin.org/status/200")
respuesta_error = requests.get("https://httpbin.org/status/404")

print("200 ok?", respuesta_ok.ok)        # True
print("404 ok?", respuesta_error.ok)      # False

# raise_for_status() lanza una excepción si el código no es 2xx
try:
    respuesta_error.raise_for_status()
except requests.exceptions.HTTPError as e:
    print(f"Error HTTP: {e}")

## 7. Timeout y errores

Siempre es buena práctica definir un **timeout** para evitar que nuestra aplicación
se quede esperando indefinidamente.

In [ ]:
# Timeout: httpbin.org/delay/{n} tarda n segundos en responder
# Ponemos un timeout de 2 segundos pero pedimos 5 de delay
try:
    respuesta = requests.get(
        "https://httpbin.org/delay/5",
        timeout=2  # segundos
    )
except requests.exceptions.ReadTimeout as e:
    print(f"Timeout alcanzado: {e}")

In [ ]:
# Error de conexión (host que no existe)
try:
    respuesta = requests.get(
        "https://servidor-que-no-existe-xyz.com",
        timeout=3
    )
except requests.exceptions.ConnectionError as e:
    print(f"Error de conexión: no se pudo conectar al servidor")

In [ ]:
# Patrón robusto: manejar múltiples tipos de error
def hacer_peticion(url, timeout=5):
    """Hace una petición GET con manejo de errores."""
    try:
        respuesta = requests.get(url, timeout=timeout)
        respuesta.raise_for_status()
        return respuesta.json()
    except requests.exceptions.Timeout:
        print(f"Timeout: el servidor no respondió en {timeout}s")
    except requests.exceptions.ConnectionError:
        print(f"Error de conexión: no se pudo conectar a {url}")
    except requests.exceptions.HTTPError as e:
        print(f"Error HTTP: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Error inesperado: {e}")
    return None

# Probamos con un endpoint válido
resultado = hacer_peticion("https://httpbin.org/get")
print("Resultado:", type(resultado))

## 8. Anatomía de un request real

Veamos **todos los componentes** de una petición HTTP completa: lo que se envía y lo que se recibe.

In [ ]:
# Hacemos una petición POST completa
respuesta = requests.post(
    "https://httpbin.org/post",
    headers={"X-Curso": "FDD-P26", "Accept": "application/json"},
    json={"mensaje": "Hola desde ITAM"},
    params={"version": "1"}
)

# === LO QUE SE ENVIÓ (Request) ===
print("=" * 50)
print("REQUEST")
print("=" * 50)
print(f"Método:  {respuesta.request.method}")
print(f"URL:     {respuesta.request.url}")
print(f"Headers:")
for k, v in respuesta.request.headers.items():
    print(f"  {k}: {v}")
print(f"Body:    {respuesta.request.body}")

In [ ]:
# === LO QUE SE RECIBIÓ (Response) ===
print("=" * 50)
print("RESPONSE")
print("=" * 50)
print(f"Status:  {respuesta.status_code} {respuesta.reason}")
print(f"Headers:")
for k, v in respuesta.headers.items():
    print(f"  {k}: {v}")
print(f"Body (JSON):")
print(json.dumps(respuesta.json(), indent=2))

In [ ]:
# Tiempo total de la petición
print(f"Tiempo de respuesta: {respuesta.elapsed.total_seconds():.3f} segundos")
print(f"Tamaño de respuesta: {len(respuesta.content)} bytes")
print(f"Encoding: {respuesta.encoding}")